# Tutorial 1: SHAP and Saliency (Feature Attribution)

In [ ]:
from utils import parity_plot

In [ ]:
from skmatter.datasets import load_csd_1000r
import numpy as np
from matplotlib import pyplot as plt
X, y = load_csd_1000r(return_X_y=True)
y = y.ravel()

## SHAP (Shapley Values)

SHAP (Shapley Additive Explanations) is a method for interpreting machine learning predictions by assigning a contribution to each input feature.

Each feature is treated as a “player” in a cooperative game, and the model prediction is treated as the “payout.”

### What SHAP tells us

- How much each feature contributes to a prediction  
- Which features increase or decrease the output  
- How different models rely on different inputs  

---

### What we will do

In this tutorial, you will:

- Train multiple models on the same dataset  
- Compute SHAP values for each model  
- Compare how different models use features  
- Connect feature importance to chemical interpretation  

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

models = {
    "Random Forest": RandomForestRegressor(),
    "Ridge": Ridge(),
    "kNN": KNeighborsRegressor()
}
n_models = len(models.keys())

fig, axes = plt.subplots(1, n_models, figsize=(n_models*4,4), sharex=True, sharey=True)

for (name, model), ax in zip(models.items(), axes):
    model.fit(X, y)
    parity_plot(y, model.predict(X), ax=ax, title=name)

In [ ]:
import shap

explainers = {
    "Random Forest": shap.TreeExplainer,
}


for (name, model) in models.items():
    if name in explainers:
        explainer = explainers[name](model)
        shap_vals = explainer.shap_values(X)

        plt.title(name)
        shap.summary_plot(shap_vals, X)


In [ ]:
# ==========================================
# TODO: compute SHAP values for a different model
# ==========================================

## Saliency Maps (Neural Network Sensitivity)

We now compute saliency maps, which measure:

$$S(x) = \frac{\partial f(x)}{\partial x}$$

This tells us how sensitive the model prediction is to each input feature.

Unlike SHAP:
- Saliency is gradient-based
- It depends strongly on model architecture

In [ ]:
import torch
import torch.nn as nn

# Convert to torch tensors
X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.float32)

# Simple neural network
net = nn.Sequential(
    nn.Linear(X.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer = torch.optim.Adam(net.parameters(), lr=0.01)

# Fast training (~5 seconds)
for _ in range(100):
    optimizer.zero_grad()
    preds = net(X_torch).squeeze()
    loss = ((preds - y_torch) ** 2).mean()
    loss.backward()
    optimizer.step()

print("Neural network trained")

In [ ]:
# Choose a datapoint
idx = 0

x = X_torch[idx].clone().detach().requires_grad_(True)

# Forward pass
y_pred = net(x)

# Backward pass
y_pred.backward()

# Saliency = gradient wrt input
saliency = x.grad.detach().numpy()
saliency = saliency / np.max(np.abs(saliency))

print("Saliency values:")
print(saliency)

In [ ]:
# ==========================================
# TODO: compute saliency for a different point
# ==========================================

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,3))
plt.bar(range(len(saliency)), saliency)
plt.title("Saliency Map (Feature Importance)")
plt.xlabel("Feature index")
plt.ylabel("Gradient")
plt.show()

## Compare SHAP vs Saliency

Questions:

- Do the same features appear important?
- Is saliency noisier?
- Does the neural network focus on different features than Random Forest?

Think about:
- SHAP = contribution
- Saliency = sensitivity